# Day 2.2 — Keyword Search Baseline

Before semantic search, build the simplest retriever we can understand:

```text
Question words → count overlap with each chunk → rank chunks
```

A baseline tells us whether a more complex solution actually helps.

## Before you begin

### Learning outcomes

Build an explainable lexical baseline and identify a meaning match it misses.

Architecture reference: [D06](../../diagrams/source/day_02.md).

### Expected observation

Exact terms rank well; a paraphrase exposes the baseline limitation.


## Concept briefing

## Establish a lexical baseline first

Keyword search is limited but valuable. It is cheap, deterministic and explainable. When
the query and document use the same words, a lexical baseline may outperform a more
complex system. It fails when the question uses a paraphrase, abbreviation or related
concept absent from the chunk.

Starting with this baseline gives semantic search something measurable to improve. If a
new embedding system is slower and no more accurate on the golden set, complexity has not
earned its place.


In [ ]:
import re, sys
from pathlib import Path
here=Path.cwd().resolve(); candidates=[here, here/"day_02_knowledge_and_state", here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src"))
from knowledge_agent.documents import load_markdown_corpus
chunks=load_markdown_corpus(project_root/"data"/"corpus")

In [ ]:
STOP={"the","a","an","is","are","of","to","for","what","which","how"}
def tokens(text):
    return {w for w in re.findall(r"[a-z0-9]+",text.lower()) if w not in STOP}
def keyword_search(question, top_k=3):
    query=tokens(question)
    ranked=sorted(chunks,key=lambda c:len(query & tokens(c.searchable_text)),reverse=True)
    return [(c,len(query & tokens(c.searchable_text))) for c in ranked[:top_k]]

In [ ]:
question="At what temperature does battery charging stop?"
for chunk,score in keyword_search(question):
    print(score, chunk.source, chunk.section)

## Break the baseline

Search for `Which equipment remains energized away from the utility grid?` The document uses related wording such as *islanded*, *critical loads*, and *remains energized*. Exact word overlap may not capture meaning well.

In [ ]:
for chunk,score in keyword_search("Which equipment remains energized away from the utility grid?"):
    print(score, chunk.source, chunk.section, "→", chunk.text[:100])

## Exercise and checkpoint

Test three questions and note where keyword search succeeds or fails. Do not call it bad—it is fast, transparent, and sometimes sufficient. Semantic embeddings add meaning-based similarity next.

## Your turn

Write one exact query and one paraphrase, then compare returned sections.

## Recap

Always establish a simple baseline before adding semantic infrastructure.
